# Week 5, Notebook 4: Spotting Hurricanes from Satellite Tiles

**A CNN for the Caribbean's biggest natural threat, plus data augmentation.**

*Caribbean AI - Adrian Dunkley*

---

### What you will build
- A CNN that reads a grayscale "satellite" tile and calls it **clear**,
  **tropical storm**, or **hurricane**.
- **Data augmentation**: the trick of showing the network shifted and flipped
  copies of the images so it stays sharp on pictures it has never seen.

### The big idea
Every summer the Caribbean watches the skies. Storms like **Hurricane Melissa**
form over warm water and can strike Jamaica, Haiti, Cuba, the Bahamas, and their
neighbours. Meteorologists study satellite images to track them. A CNN can help
by learning the tell-tale shape of a hurricane: a bright **spiral** of clouds
wrapped around a calm dark **eye**.

Our tiles are drawn with code (so no real storm is involved), but the shapes are
the real signals forecasters look for. This is also a **grayscale** problem, so
the network cannot cheat with colour; it must learn **shape**, exactly what
convolutions are best at.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load("data/satellite_tiles.npz", allow_pickle=True)
X = data["X"].astype("float32") / 255.0    # (N, 32, 32, 1), scaled to 0..1
y = data["y"]
class_names = list(data["class_names"])
print("Tiles:", X.shape, " Classes:", class_names)

In [ ]:
# Look at a few of each kind.
fig, axes = plt.subplots(3, 5, figsize=(11, 7))
for row, label in enumerate(range(3)):
    idxs = np.where(y == label)[0][:5]
    for col, idx in enumerate(idxs):
        axes[row, col].imshow(X[idx, :, :, 0], cmap="gray")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(class_names[label], fontsize=11, rotation=90)
    axes[row, 0].axis("on"); axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
plt.suptitle("Satellite tiles: clear (top), tropical storm (middle), hurricane (bottom)")
plt.show()

Notice the hurricane's **swirl and dark eye**, the storm's big fuzzy **blob**,
and the clear sky's few **scattered specks**. Those are the shapes the CNN must
learn.

### Step 1: Split, and set a baseline

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
baseline = np.bincount(y).max() / len(y)
print("Train:", len(X_train), " Test:", len(X_test))
print(f"Baseline (always guess the most common class): {baseline:.1%}")

### Step 2: A first CNN, with Dropout

We add one new layer type: **Dropout**. During training it randomly switches off
some neurons each step. That sounds destructive, but it stops the network leaning
too hard on any one clue, which helps it **generalise** to new images. Think of
it as making the network study with half its notes hidden, so it truly learns.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_cnn():
    tf.random.set_seed(0)
    m = keras.Sequential([
        layers.Input(shape=(32, 32, 1)),
        layers.Conv2D(16, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dropout(0.3),                 # randomly ignore 30% of neurons
        layers.Dense(32, activation="relu"),
        layers.Dense(3, activation="softmax"),
    ])
    m.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

plain_model = build_cnn()
plain_model.fit(X_train, y_train, validation_split=0.15,
                epochs=12, batch_size=32, verbose=0)
_, plain_acc = plain_model.evaluate(X_test, y_test, verbose=0)
print(f"Plain CNN test accuracy: {plain_acc:.1%}   (baseline {baseline:.1%})")

### Step 3: The catch - real storms are not centred

Our tiles put the storm near the middle. But a real satellite catches storms
**anywhere** in the frame. Let us test the model on **shifted** tiles (the same
storms, slid across the image) and watch accuracy fall. The model never learned
to handle that.

In [ ]:
def shift_images(imgs, dx=7, dy=5):
    # slide every image across and down (wrapping around the edges)
    return np.stack([np.roll(np.roll(im, dx, axis=1), dy, axis=0) for im in imgs])

X_test_shifted = shift_images(X_test)

# show one before and after
fig, axes = plt.subplots(1, 2, figsize=(6, 3.2))
axes[0].imshow(X_test[0, :, :, 0], cmap="gray"); axes[0].set_title("centred (training-style)")
axes[1].imshow(X_test_shifted[0, :, :, 0], cmap="gray"); axes[1].set_title("shifted (real-world)")
for a in axes: a.axis("off")
plt.show()

_, plain_shift_acc = plain_model.evaluate(X_test_shifted, y_test, verbose=0)
print(f"Plain CNN on CENTRED tiles: {plain_acc:.1%}")
print(f"Plain CNN on SHIFTED tiles: {plain_shift_acc:.1%}   <- it struggles!")

### Step 4: Data augmentation to the rescue

**Data augmentation** means making new training images by slightly changing the
ones you have: shifting, flipping, rotating. The storm is still a storm after you
slide it, so the label stays the same, but now the network sees it in many
positions and learns to spot it **anywhere**.

Keras has layers that do this automatically. We add a `RandomTranslation` and a
`RandomFlip` at the front, so every epoch the network trains on freshly jittered
tiles.

In [ ]:
augment = keras.Sequential([
    layers.RandomTranslation(0.25, 0.25, fill_mode="wrap"),  # random shifts
    layers.RandomFlip("horizontal_and_vertical"),            # random flips
], name="augmentation")

# Show four augmented copies of ONE tile, to see the variety it creates.
one = X_train[np.where(y_train == class_names.index("hurricane"))[0][0]][None, ...]
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax in axes:
    ax.imshow(augment(one, training=True)[0, :, :, 0], cmap="gray"); ax.axis("off")
plt.suptitle("Four augmented versions of the same hurricane tile")
plt.show()

In [ ]:
# Build the same CNN but with the augmentation layers in front.
tf.random.set_seed(0)
aug_model = keras.Sequential([
    layers.Input(shape=(32, 32, 1)),
    augment,                              # jitter the images during training
    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dense(3, activation="softmax"),
])
aug_model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
aug_model.fit(X_train, y_train, validation_split=0.15,
              epochs=12, batch_size=32, verbose=0)

_, aug_centred = aug_model.evaluate(X_test, y_test, verbose=0)
_, aug_shifted = aug_model.evaluate(X_test_shifted, y_test, verbose=0)
print("            centred   shifted")
print(f"Plain CNN:   {plain_acc:.0%}      {plain_shift_acc:.0%}")
print(f"Augmented:   {aug_centred:.0%}      {aug_shifted:.0%}   <- tough on shifted tiles now!")

### The lesson

Both models ace the centred tiles, but only the **augmented** one stays strong
when the storm moves. Data augmentation taught it that a hurricane is a hurricane
no matter where it sits in the frame. This is how real vision systems, including
the ones that help track Caribbean storms, are made robust for the messy real
world.

### What you learned

- A grayscale CNN can spot **hurricanes** by their **shape** (spiral + eye), a
  problem where colour cannot help, so convolutions do the work.
- **Dropout** randomly ignores neurons during training to prevent over-reliance
  and help the model generalise.
- **Data augmentation** (shifts, flips, rotations) multiplies your training data
  and makes the model robust to changes it will meet in real photos.

### Try it yourself
1. Increase the shift in `shift_images` to `dx=12, dy=10`. Does augmentation
   still hold up?
2. Add `layers.RandomRotation(0.5)` to the augmentation and retrain.
3. Remove the `Dropout` layer. Does the gap between training and validation
   accuracy change?